In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q unsloth rouge-score bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/

In [2]:
import os

# Find checkpoint-4500 for both setups
for root, dirs, files in os.walk("/kaggle/input"):
    if root.endswith("checkpoint-4500") and "adapter_config.json" in files:
        print("FOUND:", root)

# Find test data
for root, dirs, files in os.walk("/kaggle/input"):
    if "python_test_filtered.jsonl" in files:
        print("TEST:", os.path.join(root, "python_test_filtered.jsonl"))

FOUND: /kaggle/input/datasets/ayushsrivastava1805/setup-a-and-b-results-2-epochs/Setup_A_2_epochs_results/outputs_setup_a/checkpoint-4500
FOUND: /kaggle/input/datasets/ayushsrivastava1805/setup-a-and-b-results-2-epochs/Setup_B_2_epochs_results/outputs_setup_b/checkpoint-4500
TEST: /kaggle/input/datasets/ayushsrivastava1805/python-train-cot/python_test_filtered.jsonl


In [3]:
import json

ADAPTER_A = "/kaggle/input/datasets/ayushsrivastava1805/setup-a-and-b-results-2-epochs/Setup_A_2_epochs_results/outputs_setup_a/checkpoint-4500"
ADAPTER_B = "/kaggle/input/datasets/ayushsrivastava1805/setup-a-and-b-results-2-epochs/Setup_B_2_epochs_results/outputs_setup_b/checkpoint-4500"
TEST_PATH = "/kaggle/input/datasets/ayushsrivastava1805/python-train-cot/python_test_filtered.jsonl"
MAX_SEQ_LEN = 1280

test_data = [json.loads(l) for l in open(TEST_PATH)]
# test_data = test_data[:5]
print(f"Test samples: {len(test_data)}")

def strip_license_header(code_str):
    lines = code_str.split("\n")
    start = 0
    for i, line in enumerate(lines):
        s = line.strip()
        if s.startswith("#") or s == "" or s.startswith('"""') or s.startswith("'''"):
            start = i + 1
        else:
            break
    if start >= len(lines) - 2:
        return code_str
    return "\n".join(lines[start:])

def prompt_A(s):
    oldf = strip_license_header(s.get("oldf",""))[:1500]
    patch = s.get("patch","")[:800]
    return f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Review Comment:\n"

def prompt_B(s):
    oldf = strip_license_header(s.get("oldf",""))[:1500]
    patch = s.get("patch","")[:800]
    return f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Reasoning:\n"

def extract_comment_B(text):
    if "### Review Comment:" in text:
        return text.split("### Review Comment:")[-1].strip()
    return text.strip()

Test samples: 1233


In [4]:
import torch

def generate(model, tokenizer, prompt, max_new=200):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full[len(prompt):].strip()

In [5]:
from unsloth import FastLanguageModel
from tqdm import tqdm

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_A, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

preds_A = []
for s in tqdm(test_data, desc="Setup A"):
    preds_A.append(generate(model, tokenizer, prompt_A(s), max_new=150))

json.dump(preds_A, open("preds_A_2ep.json","w"))
print("Setup A done:", len(preds_A))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Starcoder2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Setup A: 100%|██████████| 1233/1233 [50:16<00:00,  2.45s/it]

Setup A done: 1233


In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_B, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

preds_B_full, preds_B_comment = [], []
for s in tqdm(test_data, desc="Setup B"):
    gen = generate(model, tokenizer, prompt_B(s), max_new=300)
    preds_B_full.append(gen)
    preds_B_comment.append(extract_comment_B(gen))

json.dump(preds_B_full, open("preds_B_full_2ep.json","w"))
json.dump(preds_B_comment, open("preds_B_comment_2ep.json","w"))
print("Setup B done:", len(preds_B_comment))

Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Starcoder2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Setup B: 100%|██████████| 1233/1233 [5:11:34<00:00, 15.16s/it]

Setup B done: 1233


In [7]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from scipy import stats
import numpy as np

actuals = [s["msg"] for s in test_data]
clean = lambda p: p if p.strip() else "."
pA = [clean(p) for p in preds_A]
pB = [clean(p) for p in preds_B_comment]

# ROUGE-L per sample
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rA = np.array([scorer.score(a,p)["rougeL"].fmeasure for a,p in zip(actuals, pA)])
rB = np.array([scorer.score(a,p)["rougeL"].fmeasure for a,p in zip(actuals, pB)])

# BERTScore per sample
_,_,FA = bertscore(pA, actuals, lang="en", verbose=True, batch_size=64)
_,_,FB = bertscore(pB, actuals, lang="en", verbose=True, batch_size=64)
FA, FB = FA.numpy(), FB.numpy()

np.save("rouge_A_2ep.npy", rA); np.save("rouge_B_2ep.npy", rB)
np.save("bert_A_2ep.npy", FA);  np.save("bert_B_2ep.npy", FB)

print("="*60)
print("RESULTS @ 4500 steps (~1.6 epochs)")
print("="*60)
for name, a, b in [("ROUGE-L", rA, rB), ("BERTScore", FA, FB)]:
    t, p = stats.ttest_rel(b, a)
    win_b = (b > a).sum(); win_a = (a > b).sum()
    print(f"\n{name}:")
    print(f"  Setup A: {a.mean():.4f}   Setup B: {b.mean():.4f}   diff: {b.mean()-a.mean():+.4f}")
    print(f"  p-value: {p:.4f}  ->  {'SIGNIFICANT' if p<0.05 else 'not significant'}")
    print(f"  B better on {win_b} samples, A better on {win_a}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/39 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/20 [00:00<?, ?it/s]

done in 15.53 seconds, 79.37 sentences/sec
RESULTS @ 4500 steps (~1.6 epochs)

ROUGE-L:
  Setup A: 0.1065   Setup B: 0.1075   diff: +0.0010
  p-value: 0.7375  ->  not significant
  B better on 576 samples, A better on 586

BERTScore:
  Setup A: 0.8514   Setup B: 0.8544   diff: +0.0030
  p-value: 0.0000  ->  SIGNIFICANT
  B better on 669 samples, A better on 564
